<a href="https://colab.research.google.com/github/jdasam/mas1004-assignment1/blob/main/assignment1_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 on Google Colab

Colab is where you run this assignment on a GPU: downloading images, training,
cleaning and checking. You do not write code here, and you do not publish the
page from here.

- You write your functions on your own computer, with your coding agent, and
  push them to your repository on GitHub. This notebook brings the code over
  from there, in section 3 and again in section 9 every time you push. Do not
  edit code in Colab. The next update replaces it with what is on GitHub.
- When training is done, you take the model files to your own computer and
  publish the page from there (section 15).
- If your computer cannot run the assignment at all (an Intel Mac, or Windows
  on an ARM processor, see README), your agent still writes the code on your
  computer, and you run the tests here after each push.

Two more things about Colab will catch you out, so deal with both before you
start:

1. Everything you save on the Colab machine is deleted when your session ends.
   Your downloaded images would go with it. So we keep the whole assignment in
   your Google Drive instead, and Colab just works on it there.
2. It gives you a plain CPU unless you ask for a GPU.

Turn the GPU on. Training ResNet18 on a CPU takes several minutes per run, and
on a GPU it is much faster. The menu at the top, `Runtime` > `Change runtime
type`, pick `T4 GPU`, then `Save`. The session restarts, so do that first.
The free GPU has a daily limit, so turn it back off when you are only looking
at pictures or writing.

## 1. What machine did we get?

In [ ]:
import torch, platform
print("python  ", platform.python_version())
print("torch   ", torch.__version__)
print("GPU     ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, running on CPU")

## 2. Connect your Google Drive

A box will pop up asking you to sign in and allow access. It is asking for your
own Drive, so that the images you download survive after this session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Bring your repository into your Drive

Put the address of your own repository in `REPO` below, the copy your agent
made at the start (README, Getting started). Not the template itself.

The first time, this copies your repository into `My Drive/mas1004/assignment1`.
After that, it makes the code there the same as your repository on GitHub now,
and prints which commit that is. Your images in `data/` and your `results/` are
not in git, so this never touches them. Anything else you changed in that
folder by hand is replaced.

If that folder was made from another address, for example from the template in
class on 29 September, it is renamed to `assignment1_old_...` and your
repository is copied in fresh. The `data/` and `results/` folders of the old
one are moved into the new one, so you keep your images.

If it stops with "could not read Username", your repository is private. Make it
public on GitHub, under Settings, and run the cell again.

In [ ]:
import os
import shutil
import subprocess
import time
from pathlib import Path

# Your own repository, for example "https://github.com/hong-gildong/my-classifier"
REPO = "https://github.com/YOUR-NAME/YOUR-REPOSITORY"

TEMPLATE = "https://github.com/jdasam/mas1004-assignment1"
HOME = Path("/content/drive/MyDrive/mas1004")
PROJECT = HOME / "assignment1"


def git(*args):
    done = subprocess.run(["git", *args], capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{done.stderr}")
    return done.stdout


def address(url):
    return url.strip().rstrip("/").removesuffix(".git").lower()


def bring_code():
    """Make the code in PROJECT the same as your repository on GitHub."""
    git("-C", str(PROJECT), "fetch", "-q")
    git("-C", str(PROJECT), "reset", "-q", "--hard", "@{u}")
    print(git("-C", str(PROJECT), "log", "-1",
              "--format=code from your commit %h (%cr): %s"))


if "YOUR-NAME" in REPO or address(REPO) == address(TEMPLATE):
    raise ValueError("Put the address of your own repository in REPO, "
                     "then run this cell again.")

# git refuses to work in a folder it thinks belongs to another user, and folders
# in Drive can look like that.
git("config", "--global", "--add", "safe.directory", "*")

HOME.mkdir(parents=True, exist_ok=True)
old = None
if PROJECT.exists():
    try:
        here = address(git("-C", str(PROJECT), "remote", "get-url", "origin"))
    except RuntimeError:
        here = None
    if here != address(REPO):
        old = HOME / time.strftime("assignment1_old_%m%d_%H%M")
        PROJECT.rename(old)
        print(f"{PROJECT} came from {here}, so it is now {old}")

if not PROJECT.exists():
    git("clone", "-q", REPO.strip(), str(PROJECT))
    print(f"copied {REPO.strip()} into {PROJECT}")
    for name in ("data", "results"):
        if old is not None and (old / name).exists():
            shutil.move(str(old / name), str(PROJECT / name))
            print(f"moved {name}/ over from {old}")

bring_code()
os.chdir(PROJECT)
print("working in", Path.cwd())

## 4. Install what Colab does not already have

PyTorch is already there. This adds the image downloader and the ONNX tools
that put your model on the web page. Colab forgets them when the session ends,
so run this every time you open the notebook.

In [ ]:
!pip install -q -r requirements.txt

## 5. Check that the given code works

Six tests should pass. These cover code you were given, not code you write.

In [ ]:
!python -m pytest -q tests/test_export.py

## 6. Download your images

Change the search phrases to your own categories. This writes into
`data/raw` inside your Drive, so it is still there tomorrow.

If you already downloaded images on your own computer, do not download them
again here. Upload your `data` folder into `My Drive/mas1004/assignment1/` in
Google Drive instead, so that you keep the same images. Either way, from here
on your images live in Drive, and you clean them here, in section 11.

In [ ]:
!python src/collect.py --classes "espresso cup,wine glass,paper coffee cup" --n 150
if not Path("data/clean").exists():
    !cp -r data/raw data/clean

## 7. Look at every image

This draws every image onto sheets. Look at all of them. Do not remove anything
yet: your first training run is on the images as they came.

In [ ]:
!python src/clean.py look
from IPython.display import Image, display
for sheet in sorted(Path("results/cleaning/look").glob("*.png")):
    print(sheet.name)
    display(Image(filename=str(sheet)))

## 8. Put your images from a new source in

Make a folder `data/new_images/<category>` for each category, with exactly the
same names as the folders in `data/clean`, and put at least five images into
each. They must come from a source you can be sure is not in your downloads,
such as photos you took yourself or a friend took, or frames from a video you
recorded, or images from the web that you are sure your download could not
have found. Save them as JPEG or PNG: the code here cannot read the HEIC files
an iPhone makes by default. The file browser on the left can upload
them, under `drive/MyDrive/mas1004/assignment1/data`, or you can drop them
into the same folder in Google Drive from your phone.

The second cell checks that none of them is a near copy of a downloaded image.

In [ ]:
for folder in sorted(Path("data/clean").iterdir()):
    mine = Path("data/new_images") / folder.name
    mine.mkdir(parents=True, exist_ok=True)
    print(f"{mine}   {len(list(mine.iterdir()))} images")

In [ ]:
!python src/clean.py overlap | tee results/overlap.txt
if Path("results/cleaning/overlap.png").exists():
    display(Image(filename="results/cleaning/overlap.png"))

## 9. Bring in your code and test it

Write the functions in `src/data.py`, `src/train.py` and `src/evaluate.py` on
your own computer, with your agent, and run the tests there. Then commit and
push. The first cell below brings what you pushed into your Drive and prints
which commit it is. Check that it is the one you just pushed.

Run it again every time you push a change. Do not edit the code here: the next
time it runs, anything changed here is replaced with what is on GitHub.

The second cell runs the tests here too, so you can see that Colab has the same
code. The first run downloads the ImageNet weights. If your computer cannot run
the assignment, this is the only place you run the tests.

In [ ]:
# bring_code comes from the cell in section 3. If Colab says it is not defined,
# run sections 2 and 3 again first.
bring_code()

In [ ]:
!python -m pytest -q tests

## 10. Train, before cleaning anything

In [ ]:
!python src/run.py
!python src/run.py --scratch --tag scratch
!python src/run.py --freeze --lr 1e-3 --tag frozen

In [ ]:
for name in ["run_curves.png", "run_confusion.png", "run_worst.png"]:
    if Path(f"results/{name}").exists():
        print(name)
        display(Image(filename=f"results/{name}"))

## 11. Clean your data

Write your rule first. Then ask the rest of your data which images look least
like their class, and which are near copies of each other.

In [ ]:
!python src/clean.py suspects
for sheet in sorted(Path("results/cleaning/suspects").glob("*.png")):
    print(sheet.name)
    display(Image(filename=str(sheet)))

Remove what breaks your rule, with the reason. Repeat this cell as many times
as you need, with your own file names and reasons.

In [ ]:
!python src/clean.py remove wine_glass/0063.jpg wine_glass/0069.jpg --reason "chart, not a photo"

In [ ]:
!python src/clean.py count | tee results/clean_count.txt

## 12. Train on the clean data and compare on your new images

In [ ]:
!python src/run.py --tag clean
!python src/check.py --compare run clean | tee results/compare.txt

## 13. Export the clean model and check your work

This writes the model the page needs into `docs/` and runs the checker. Copy
the whole output of `check.py` into your long report.

`check.py` warns that `docs/model.onnx` is not committed. That is expected
here: you commit it on your own computer, in section 15.

In [ ]:
!python src/export_web.py --tag clean
!python src/check.py | tee results/check.txt

## 14. Try the demo page without leaving Colab

This serves the `docs/` folder and opens it in a panel. The upload and drawing
tabs work here. The webcam does not, because of how Colab sandboxes the page,
so test the webcam on your own machine or on your published page.

In [ ]:
import subprocess, time
from google.colab import output

server = subprocess.Popen(["python", "-m", "http.server", "-d", "docs", "8000"])
time.sleep(2)
output.serve_kernel_port_as_window(8000)

## 15. Take the model to your own computer and publish it there

Colab has done its part. The page is published from your own computer, not from
here.

The cell below packs what you need into `to_my_computer.zip` and downloads it:
the three files the page needs, from `docs/`, and the pictures, settings and
saved outputs in `results/` for your long report. The trained models themselves (`results/*.pt`,
45 MB each) are left out. If the download does not start, the zip is also in
your Drive, in `My Drive/mas1004/assignment1/`.

Then, on your own computer:

1. Unzip it. Copy `model.onnx`, `model.json` and `selftest.json` from its
   `docs` folder into the `docs` folder of your repository, next to
   `index.html` and `app.js`.
2. Look at the page with `uv run python -m http.server -d docs 8000` and
   http://localhost:8000, and check that the badge is green. If uv does not run
   on your computer, `python3 -m http.server -d docs 8000` does the same.
3. Commit and push, and turn on GitHub Pages, as in README, Problem 6.

The pictures from `results/` go into your report, not into git.

In [ ]:
!rm -f to_my_computer.zip
!zip -qr to_my_computer.zip docs/model.onnx docs/model.json docs/selftest.json results -x "results/*.pt" "results/cleaning/features.npz"
from google.colab import files
files.download("to_my_computer.zip")